# Day 5: Meeting Minutes Creator

> This notebook documents **Day 5 only**.  
> Later days have separate notebooks.

## Overview

This notebook builds an end-to-end AI workflow that combines audio transcription with LLM-powered text analysis to create structured meeting minutes.

## Learning Objectives

- Integrate Google Drive with Colab for file access
- Use automatic speech recognition (ASR) for audio transcription
- Compare open-source (Whisper) vs. OpenAI transcription options
- Build end-to-end pipeline: Audio → Transcription → LLM Analysis → Meeting Minutes
- Generate structured meeting minutes with summaries, discussion points, takeaways, and action items
- Understand the complete workflow from raw audio to formatted output

## Resources

- [Meeting Minutes Colab](https://colab.research.google.com/drive/1KSMxOCprsl1QRpt_Rq0UqCAyMtPqDQYx?usp=sharing)
- [Denver City Council Audio Extract](https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing)
- [HuggingFace Whisper Models](https://huggingface.co/models?pipeline_tag=automatic-speech-recognition)
- [OpenAI Audio API](https://platform.openai.com/docs/guides/speech-to-text)

## Setup

### Install Dependencies

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate

### Imports

In [ ]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, pipeline
import torch

### Constants

In [ ]:
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

### Mount Google Drive

Connect this Colab to your Google Drive to access audio files.

**Instructions:**
1. Download the Denver City Council audio extract: https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing
2. Create a folder called `llms` in your Google Drive
3. Upload the file as `denver_extract.mp3` to that folder
4. Or record your own audio and place it in the same location

In [ ]:
drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/llms/denver_extract.mp3"

### HuggingFace Authentication

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

### Open Audio File

In [ ]:
audio_file = open(audio_filename, "rb")

## Step 1: Transcribe Audio

We have two options for transcription:

1. **Open Source (Whisper)** - Free, runs on GPU, good quality
2. **OpenAI API** - Paid, cloud-based, excellent quality

### Option 1: Use Open Source for Transcription - Hugging Face Pipelines

In [ ]:
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_filename)
transcription = result["text"]
print(transcription)

open_source_transcription = transcription

### Option 2: Use OpenAI for Transcription

In [ ]:
# Sign in to OpenAI using Secrets in Colab
AUDIO_MODEL = "gpt-4o-mini-transcribe"

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

### Compare Both Transcriptions

In [ ]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

## Step 2: Analyze & Report

Now we use an LLM to transform the raw transcription into structured meeting minutes.

### Define System Message and User Prompt

In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
]

### Configure Quantization

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

### Load Model and Generate Meeting Minutes

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

response = tokenizer.decode(outputs[0])

display(Markdown(response))

## Key Learnings

- **End-to-End Workflows:** Combining multiple AI capabilities (ASR + LLM) creates powerful applications
- **Transcription Options:** Open-source (Whisper) vs. OpenAI API - tradeoffs between cost, quality, and control
- **Google Drive Integration:** Colab can mount Drive for persistent file storage
- **Structured Output:** LLMs can transform unstructured transcripts into structured meeting minutes
- **System Prompts:** Clear system prompts guide LLM behavior for specific output formats
- **Real-World Application:** This pattern applies to any audio-to-structured-text workflow (interviews, lectures, podcasts)